**Go** è un linguaggio progettato per facilitare la costruzione di software semplice, affidabile e scalabile.  
Queste caratteristiche sono particolarmente importanti nei sistemi distribuiti, dove tocca gestire tante attività contemporaneamente e programmi/comunicazioni di rete che devono funzionare anche al crescere del carico. 

Go nasce nel 2007 in Google, i suoi obiettivi principali sono:
- **efficienza del software** durante l'esecuzione
- **semplicità di sviluppo**

Per quanto riguarda il primo obiettivo Go è **linguaggio compilato** e **tipizzato staticamente** (i.e. i tipi delle variabili sono verificati prima dell'esecuzione); per il secondo sintassi semplice e nell'esperienza d'uso che sembri più simile a un linguaggio interpretato.

Due riferimenti importanti per Go sono **C** e **CSP**. 

Dal C derivano elementi come la sintassi delle espressioni, i costrutti per il controllo del flusso come `if`, `for`, `switch`, tipizzazione base e *puntatori*, passaggio di parametri per valore (i.e. la funzione riceve copia del valore passato come argomento) e l'attenzione all'efficienza in fase di esecuzione.  

Il CSP (Communicating Sequential Processes) è invece un modello per descrivere sistemi costituiti da attività sequenziali, che procedono in modo concorrente e comunicano tra loro.  
Di sto CSP bisogna ricordare quest'idea chiave: **all'interno di ogni attività le istruzioni seguono un ordine, ma più attività possono avanzare concorrentemente e coordinarsi tramite comunicazione**; da qui deriva la base delle **goroutine** e dei **canali** in Go.

Go viene introdotto per tre ragioni: prima cosa la **distribuzione del programma** in quanto Go può produrre un singolo eseguibile binario che nelle configurazioni adatte contiene tutto il necessario per l'esecuzione --> facilita il trasferimento e l'avvio del software sulle macchine destinazione.  
La seconda è il **supporto alla programmazione di rete**, in quanto Go dispone di librerie necessarie ad utilizzare meccanismi come RPC e gRPC.  
Il terzo elemento è il **modello di concorrenza**, basato su **goroutine** (attività concorrenti leggere), **channel** (canali attraverso cui le attività comunicano) e **select** (costrutto che permette di attendere su più canali).

È il runtime di Go ad occuparsi della gestione delle goroutine e di distribuirne l'esecuzione sui thread SO --> *una goroutine non corrisponde necessariamente a un thread dedicato*.

Go offre anche **struct**, **metodi** e **interfacce** ma non nel senso classico OOP, quanto più come strumenti per organizzare il codice e definire comportamenti (i.e. non c'è ereditarietà, ma solo composizione).

Infine Go ha anche un meccanismo di **garbage collection**.

#### Esempio
Vediamo un esempio per iniziare a capire la sintassi di Go:

<img src="imgs/go1.png" alt="go1" width="400"/>

Qui il programma avvia tre attività concorrenti e raccoglie i msg attraverso un canale (stiamo simulando tre risposte da server).  
Con `ch := make(chan string, 3)` si ha `chan string` che indica un canale che trasporta valori di tipo stringa, `make` che crea e inizializza il canale, `3` che indica che il canale contiene un buffer che può contenere 3 valori e `:=` dichiara una variabile e la inizializza, facendo dedurre al compilatore il tipo.

Il canale è creato prima delle goroutine affinché lo possano usare per comunicare col `main`. 

Il primo `for` esegue tre iterazioni con `i` da 1 a 3 (sintassi uguale a C ma no parentesi tonde); la parola chiave `go` davanti a una funzione dice che deve essere eseguita come goroutine --> il `main` può proseguire senza aspettare che quella chiamata termini.  
In questo caso si usa una funzione anonima `func(id int)`.  
Ciascuna goroutine attende 100 ms con la sleep, costruisce una stringa con `fmt.Sprintf` e la invia al canale con `ch <- msg`.

Qui importante il senso della freccia: `ch <- msg` significa "invia il valore `msg` al canale `ch`", mentre `r := <-ch` significa "ricevi un valore dal canale `ch` e assegnalo a `r`".

Dopo aver avviato le goroutine, il `main` esegue un secondo `for` dove ad ogni iterazione, `select` attende che possa procedere uno dei due casi:
- `case r := <-ch:` se arriva un valore dal canale `ch`, lo assegna a `r` e stampa il messaggio ricevuto
- `case <-time.After(time.Second):` se non arriva alcun valore dal canale entro un secondo, stampa un messaggio di timeout (`time.After` fornisce un canale sul quale arriva una notifica dopo il tempo specificato).

Chiaramente l'ordine delle risposte non è garantito: avviare le tre goroutine in ordine non obbliga a ricevere i msg nello stesso ordine.

#### Esempio Hello World
```go
package main
import "fmt"

func main() {
    fmt.Println("Hello, 世界!")
}
```
`package main` indica che il file fa parte del package `main`, ogni package può contenere più file `.go` della stessa dir, come in Java.  
Vabbé l'import per usare il package `fmt` che offre le funzioni per input e output formattati, e la funzione `main` che stampa l'hello world. 

La presenza di caratteri cinesi evidenzia il supporto a Unicode, e si nota anche l'assenza di ; alla fine delle istruzioni che è opzionale.

#### Compilazione ed esecuzione; packages
Go ha una propria **toolchain**, i.e. ha strumenti integrati per compilare ed eseguire il codice:
- `go run helloworld.go` compila ed esegue il file `helloworld.go`
- `go build helloworld.go` compila il file e produce un eseguibile binario nella stessa dir
- `go install helloworld.go` compila il file e produce un eseguibile binario nella destinazione prevista dalla toolchain

Le prime due sono quelle che ci interessano di più, la distinzione è che con `go run` compilazione e avvio avvengono svolti insieme senza lasciare nella directory il normale eseguibile prodotto da `go build` --> sempre e comunque linguaggio compilato.

Un **package** può contenere dichiarazioni di tipi, funzioni, variabili e costanti. Una regola fondamentale di Go è che **l'iniziale del nome determina se un identificatore è esportato**, i.e. accessibile al'esterno del package: **se l'iniziale è maiuscola l'identificatore è esportato, altrimenti no**.

Da qui il motivo della P in `fmt.Println`: la funzione deve essere esportata dal package `fmt` per poter essere utilizzata dal nostro codice.

es. `math.pi` con `math` package importato produce errore perché tutte le funzioni utilizzabili fuori dal package iniziano sempre con lettera maiuscola!

Si possono raggruppare più import in un blocco:
```go
import (
    "fmt"
    "math"
)

func main() {
    fmt.Printf("now you have %g problems.\n", math.Sqrt(7))
}
```
Differenza tra `Println` e `Printf` (entrambe package `fmt`): la prima stampa il messaggio e va a capo, la seconda stampa il messaggio senza andare a capo e permette di formattare l'output con specificatori di formato come `%d`, `%s`, `%f`, `%g` ecc.. per andare a capo in sto caso esplicita `\n`

#### Caratteristiche generiche

Altra cosa importante in Go è che **il tipo segue il nome del parametro**, al contrario di C/C++/Java:

```go
func add(x int, y int) int {
    return x + y
}
```
oppure si può scrivere in forma compatta per più parametri dello stesso tipo:
```go
func add(x, y int) int {
    return x + y
}
```

Altra caratteristica è che **una funzione può restituire più risultati distinti**:
```go
func swap(x, y string) (string, string) {
    return y, x
}
```

In Go inoltre la gestione degli errori è esplicita in quanto molte funzioni Go **restituiscono oltre al risultato dell'operazione anche un valore che descrive un eventuale errore** (del tipo `risultato, erro := funzioneChePuòFallire()`).  
Il chiamante poi controlla `err`, e se `err == nil`  --> nessun errore, altrimenti bisogna gestire l'errore.

<img src="imgs/go2.png" alt="go2" width="300"/>

Con `data, err := os.ReadFile("example.txt")` si legge il contenuto del file restituendo `data` come i dati letti rappresentati come slice (la vediamo dopo) di byte, di tipo `[]byte`, e `err` come eventuale errore.  

Altra funzione importante che vedremo è `defer`, che permette di posticipare l'esecuzione di una funzione fino al termine della funzione corrente.  
Utilissima ad esempio per chiudere file o connessioni di rete: di solito ci si dimentica, ma con `defer` possiamo scriverlo subito dopo l'apertura del file/connessione e siamo sicuri che verrà eseguito solo alla fine della funzione, anche in caso di errori.


Go consente anche di assegnare un nome ai risultati direttamente nella firma della funzione:
```go
func split(sum int) (x, y int) {
    x = sum * 4 / 9
    y = sum - x
    return
}
```
`x` e `y` sono variabili intere già dichiarate nella firma! Il return è quindi scritto senza espressioni, si parla d i**naked return**.

In Go, la parola chiave `var` indica una o più variabili. Come per i parametri di funzioni, il tipo è scritto dopo il nome, ad es. `var i int` dichiara una variabile intera chiamata `i`.  

Una differenza importante rispetto alle variabili locali non inizializzate in è che **Go assegna sempre un valore iniziale**: se non lo specifichiamo viene inizializzato lo **zero value** che dipende dal tipo:
- tipi numerici --> `0`
- booleani --> `false`
- stringhe --> `""` (stringa vuota)
- puntatori, funzioni, interfacce, slice, canali e mappe --> `nil` (i.e. non puntano a nulla)

<img src="imgs/go3.png" alt="go3" width="400"/>

`var c, python, java bool` dichiara tre variabili ooleane **a lvl di package**, in quanto si trova fuori dalle funzioni --> utilizzabili tutte dal codice nello stesso package.  
`var i int` invece sta nel `main` --> **variabile locale alla funzione**.  
Poiché nessuna delle variabili ha inizializzatore esplicito, la stampa produrrebbe `0 false false false` (i.e. zero value per ciascun tipo).

Se c'è un'assegnazione iniziale, il **tipo può essere dedotto** e quindi non serve scriverlo esplicitamente: `var count = 10` invece di `var count int = 10`.
Dentro le funzioni si può usare anche la forma compatta `count := 10` senza scrivere `var` né specificare il tipo.

Altro caso di inferenza del tipo:
```go 
var i int
j := i
```
il compilatore deduce che `j` è di tipo `int` perché `i` è di tipo `int`.

La distinzione importante è che `:=` **dichiara e inizializza** una nuova variabile, mentre `=` **assegna** un nuovo valore a una variabile già dichiarata.

Tra i tipi base offerti da Go:
- `bool` 
- `string` (Unicode!)
- `int` interi con segno
- `uint` interi senza segno
- `float32`, `float64` numeri in virgola mobile 

Un aspetto importante è che Go **richiede conversioni esplicite tra valori con tipi numerici diversi**: ad es:
```go
var i int = 42
var f float64 = float64(i)
```
per cui `float64(i)` converte `i` in un valore di tipo `float64`, ma non si cambia il tipo di `i`, si produce un nuovo valore `f` con valore pari alla conversione di `i` --> passaggio esplicito.  

#### Controllo del flusso e altre operazioni simpatiche
**Ciclo for**

<img src="imgs/go4.png" alt="go4" width="300"/>

Go ha solo il `for` come costrutto per il ciclo, ma può essere usato in tre forme diverse.  

La prima forma è quella in figura: `for i:= 0; i < 10; i++` è simile a C/C++/Java, con inizializzazione, condizione e aggiornamento. Nel corpo `sum += i` equivale al solito a `sum = sum + i`.  
Rispetto al C no parentesi tonde attorno alle tre componenti, ma le parentesi graffe del corpo restano obbligatorie.

La seconda è la forma come `while` e la terza come ciclo infinito

<img src="imgs/go5.png" alt="go5" width="300"/>

`for sum < 1000 {` è equivalente a `while sum < 1000 {` in C/C++/Java, mentre `for {` è un ciclo infinito come `while(true)` (che può essere interrotto solo con un `break` o un `return`).  
Nell'esempio si arriva a sum pari a 512, la condizione `sum < 1000` è vera, quindi si entra nel ciclo e si raddoppia sum --> la condizione è falsa e quindi sum restituisce 1024 --> il valore finale può superare la soglia perché la condizione è controllata PRIMA di eseguire il corpo.

es.

<img src="imgs/go6.png" alt="go6" width="400"/>

Il programma legge gli argomenti ricevuti in riga di comando e li stampa separati da spazi. Gli argomenti sono disponibili in `os.Args`, che è uno slice di stringhe (vedi dopo, per ora vedila come sequenza indicizzata).  
In `os.Args[0]` c'è il percorso dell'eseguibile. Quelli forniti dall'utente partono da `os.Args[1]`, per questo il ciclo parte da `i := 1` e finisce a `i < len(os.Args)`.

Ad ogni iterazione la variabile stringa `s` accumula il testo, mentre `sep` contiene lo spazio da stampare prima di ciascun argomento --> `s += sep + os.Args[i]` concatena lo spazio e l'argomento corrente.

**Condizionali**: 

`if` ed `else` come al solito, `else if` introduce ulteriori condizioni.  
Tutto è controllato in ordine e viene eseguito il primo ramo con condizione vera. 

Come per il `for` non servono parentesi tonde attorno alla condizione, ma le parentesi graffe del corpo restano obbligatorie. Piccola rottura di zebedei è che `else` va scritto sempre sulla stessa riga della graffa che chiude il ramo precedente: `} else {` altrimenti produce errore di sintassi.

Una particolarità è che si può definire **un'istruzione iniziale prima della condizione, separandole con** `;`.  
Questa istruzione viene eseguita prima della verifica ed è spesso usata per dichiarare una variabile necessaria solo alla scelta condizionale, es:

```go
func pow(x, n, lim float64) float64 {
    if v := math.Pow(x, n); v < lim {
        return v
    } else {
        fmt.Printf("%g >= %g\n", v, lim)
    }
        // can't use v here
    return lim
}
```
La funzione calcola $x^n$ e lo confronta con `lim`. Se il risultato è minore di `lim` lo restituisce, altrimenti stampa un messaggio e restituisce `lim`.  
In `if v := math.Pow(x, n); v < lim {` anzitutto si calcola `v` e poi si verifica la condizione.

Nota come `v` sia visibile solo all'interno del ramo `if` e del ramo `else`, ma non dopo il blocco condizionale.

Per selezionare tra più alternative si usa `switch`:

<img src="imgs/go7.png" alt="go7" width="350"/>

Il terzo ramo ha `Saturday, Sunday:` separando i valori con una virgola come si vede significa che **quel ramo vale per ciascuno dei valori elencati**.

La differenza centrale rispetto al C è che **Go esegue solo il ramo selezionato**, senza proseguire automaticamente nei rami successivi --> non serve il `break` alla fine di ciascun caso.  
Si può inoltre aggiungere `default`, eseguito quando nessun caso corrisponde.

**defer**

Anticipato prima, permette di **registrare una chiamata da eseguire quando la funzione che la contiene sta per terminare** --> utile per la pulizia di risorse (tipo chiudere file o connessioni; o rilasciare lock).

Se si hanno più `defer` allora **l'ordine è LIFO**, l'ultima registrata sarà la prima a essere eseguita.

<img src="imgs/go8.png" alt="go8" width="400"/>

**Puntatori**

Un puntatore identifica l'indirizzo di una variabile in memoria. Quindi a solito distinguiamo il **contenuto** della variabile, il suo **indirizzo** in memoria e il **puntatore** che contiene quell'indirizzo e permette quindi di accedere alla variabile.  

I due operatori fondamentali sono:
- `&i` che ottiene l'indirizzo della variabile `i`
- `*p` che accede al contenuto della variabile puntata da `p` 

```go
var p *int
i := 1
p = &i // p contiene l'indirizzo di i
fmt.Println(*p) // stampa il contenuto della variabile puntata da p, cioè 1
*p = 2 // cambia il contenuto della variabile puntata da p, i.e. equivale a i = 2
fmt.Println(i) // stampa 2
```
Abbiamo però detto che Go passa i parametri **per valore**: i.e. se passiamo un intero allora la funzione ne riceve una copia.

Se passiamo un puntatore allora, riceve la copia del puntatore, ma quella continua a puntare alla stessa variabile --> la funzione può quindi modificare il contenuto della variabile puntata tramite la dereferenziazione (accesso diretto al valore memorizzato).  
Ciò spiega *come una funzione possa modificare i dati del chiamante pur mantenendo il passaggio per valore*.

**Differenze rispetto a C sui puntatori**:
1. In Go **non ci sono aritmetiche sui puntatori**, quindi non si può fare `p++` per spostarsi alla cella successiva di un array.
2. Il valore zero di un puntatore è `nil`. Tentare di dereferenziare un puntatore `nil` genera un *panic* (errore di runtime). 
3. In Go **è sicuro restituire l'indirizzo di una variabile locale**.  
Mentre infatti in C restituire l'indirizzo di una variabile locale automatica lascia un indirizzo non più valido dopo la fine della funzione, in Go la durata della memoria viene gestita affinché la variabile possa continuare a esistere e sia ancora raggiungibile.  
Il compilatore analizza se una variabile deve sopravvivere alla chiamata tramite la cosiddetta *escape analysis* e può collocarla nell'**heap** anziché salvarla nello spazio temporaneo della chiamata (lo **stack**).  
Il garbage collector si occuperà poi di liberare la memoria quando non ci sono più riferimenti a quella variabile.

#### Tipi aggregati: struct, array, slice e mappe
I tipi aggregati permettono di organizzare insieme più dati.

Una **struct** raggruppa campi, ciascuno con un nome e un tipo. Ogni campo può avere tipi diversi (es. una persona può avere nome di tipo stringa, età di tipo int, altezza di tipo float64).

```go 
type Vertex struct {
    X int
    Y int
}
v := Vertex{1, 2} 
```
`type Vertex struct` definisce un nuovo tipo chiamato `Vertex`, composto da due campi interi `X` e `Y`.  
I campi sono accessibili con la notazione puntata: `v.X` e `v.Y`.

Si può anche ottenere un puntatore alla struct con `p := &v`: G opermette di scrivere direttamente `p.X` invece di `(*p).X` per accedere al campo attraverso il puntatore.

Un **array** contiene tutti elementi dello stesso tipo e ha lunghezza fissa.  
Con ad es. `var a [3]string` si dichiara un array di 3 stringhe, inizializzate a `""` (zero value). Gli indici partono da 0, quindi `a[0]`, `a[1]` e `a[2]` sono gli elementi validi.  
Nella pratica, invece degli array, si usano più spesso gli **slice** in quanto sono strutture più flessibili e dinamiche.

Una **slice** rappresenta una porzione di un array sottostante, a differenza di un array però la sua lunghezza può cambiare. La notazione distingue i due tipi:
- `[6]int` è un array di 6 interi
- `[]int` è uno slice di interi, senza lunghezza fissa

es.
```go
primes := [6]int{2, 3, 5, 7, 11, 13} // array di 6 interi
var s []int = primes[1:4] // slice che contiene gli elementi da indice 1 a 3 dell'array, quindi [3, 5, 7]
```
La regola è sempre **estremo iniziale incluso, estremo finale escluso**.  
Si può creare una slice sia da un array che da un'altra slice.

Attenzione al fatto che **creare una slice non copia i dati**, ma crea un riferimento all'array sottostante! Quindi assegnare tipo `s[0] = 42` implica modificare `primes[1]` che è l'elemento dell'array sottostante.  
Quindi due slice costruite sotto lo stesso array possono condividere parte dei dati: una modifica effettuata attraverso una può essere visibile anche attraverso l'altra.

Per capire lo slice bisogna distinguere tra **lunghezza** e **capacità**:
- `len(s)` restituisce il numero di elementi attualmente presenti nello slice
- `cap(s)` restituisce il numero di elementi dell'array sottostante a partire dall'inizio dello slice, quindi la capacità massima dello slice prima che sia necessario allocare un nuovo array sottostante. Accedere a un indice non valido, oltre la capacità dello slice, genera un *panic* durante l'esecuzione.

Le slice possono anche essere create con la funzione `make`:
```go
s := make([]int, 5, 10) // slice di 5 interi inizializzati a zero su array sottostante di capacità 10
```
Fare `make([]int, 0, 5)` crea una slice con lunghezza 0, capacità 5 e nessun elemento attualmente presente in slice.  

Per aggiungere elementi a uno slice si usa la funzione `append`:
```go
s = append(s, 1) // aggiunge 1 alla slice s
```
In Go esistono **funzioni variadiche**, i.e. accettano un numero variabile di argomenti dello stesso tipo, e append è una di queste:
```go
s = append(s, 2, 3, 4) // aggiunge 2, 3 e 4 alla slice s
```
Quando si fa `append`, se la capacità sottostante non basta, Go:
1. alloca un array più grande
2. copia i vecchi elementi nello slice
3. aggiunge i nuovi elementi
4. restituisce una slice riferita al nuovo array
   
Eventuali altre slice continueranno a riferirsi al vecchio array.

**Mappe**

Una **mappa** associa una chiave a un valore. Il tipo si scrive `map[KeyType]ValueType`, ad es. `map[string]int` è una mappa che associa stringhe a interi.  

Le operazioni principali sono:
- `m[key] = element` per inserire o aggiornare il valore associato a `key`
- `element = m[key]` per leggere il valore associato a `key`
- `delete(m, key)` per rimuovere la coppia chiave-valore associata a `key`
- `element, ok = m[key]` per leggere il valore associato a `key` e verificare se la chiave esiste nella mappa (ok è true se la chiave esiste, false altrimenti)

`range`, usato insieme a `for`, permette di attraversare diverse strutture dati. Per array e slice fornisce **indice e valore**, per una mappa fornisce **chiave e valore**.

Esempio `range` con slice:
```go
package main
import "fmt"

var pow = []int{1, 2, 4, 8, 16, 32, 64, 128}

func main() {
    for i, v := range pow {
        fmt.Printf("2**%d = %d\n", i, v)
    }
}
```
`pow` è slice contenente le potenze di 2, nel ciclo `for i, v := range pow` ad ogni iterazione assegna a `i` l'indice e a `v` il valore corrispondente.  
La stampa produce righe come `2**0 = 1`, `2**1 = 2`, ecc.. fino a `2**7 = 128`.

Esempio `range` con mappa:
```go
func main() {
    nums := []int{2, 3, 4}
    sum := 0
    for _, num := range nums {
        sum += num
    }
    fmt.Println("sum:", sum)
    for i, num := range nums {
        if num == 3 {
            fmt.Println("index:", i)
        }
    }
    kvs := map[string]string{"a": "apple", "b": "banana"}
    for k, v := range kvs {
        fmt.Printf("%s -> %s\n", k, v)
    }
    for k := range kvs {
        fmt.Println("key:", k)
    }
}
```
Il primo ciclo somma tutti gli elementi della slice `nums`: dato che l'indice non serve, si usa `_` (detto **blank identifier**) per ignorarlo (se non lo si fa errore di compilazione, in Go scrivere una variabile locale e non usarla infatti può generare errore).

Il secondo ciclo usa sia indice che valore per individuare dove compare il 3.

I cicli sulla mappa mostrano come ottenere coppie chiave-valore e come ottenere solo le chiavi. Attenzione al fatto che **l'ordine di iterazione su una mappa non è garantito**: ogni volta che si esegue il programma, l'ordine può cambiare.

Per le stringhe, `range` attraversa i caratteri Unicode, restituendo l'indice del primo byte di ciascun carattere e il valore del carattere stesso (di tipo `rune`, alias per `int32`).

Esempio di WordCount con mappa e range:
```go
package main
import (
    "fmt"
    "strings"
)
func WordCount(s string) map[string]int {
    counts := make(map[string]int)
    words := strings.Fields(s)
    for _, w := range words {
        counts[w]++
    }
    return counts
}
func main() {
    result := WordCount("I am a student and I am learning Go")
    fmt.Println(result)
}
```
WordCount riceve una str e restituisce una mappa che associa ciascuna parola al numero di occorrenze.

Prima con `make` crea la mappa dei contatori, poi `strings.Fields(s)` divide la stringa in parole separate da spazi, range attraversa le parole e `counts[w]++` incrementa il contatore per ciascuna parola.  
Alla prima comparsa di una parola non serve inizializzare esplicitamente il contatore in quanto una chiave assente restituisce `0`.

#### Funzioni anonime, closure
Una **funzione anonima** è una funzione senza nome definita direttamente dove serve. Es `func(x, y int) int { return x + y }` è una funzione anonima che somma due interi.

Una **closure** è una funzione anonima che può accedere a variabili definite al di fuori della funzione stessa. Il punto importante è che queste variabili possono conservare il loro stato tra chiamate successive, anche quando la funzione esterna ha già terminato.

Closure: es. 1
```go
package main
import "fmt"

func intSeq() func() int {
    i := 0
    return func() int {
        i++
        return i
    }
}

func main() {
    nextInt := intSeq()
    fmt.Println(nextInt()) // 1
    fmt.Println(nextInt()) // 2
    fmt.Println(nextInt()) // 3

    newInts := intSeq()
    fmt.Println(newInts()) // 1
}
```
L'ouput con `go run closure.go` è `1 2 3 1`.  
La funzione `intSeq` restituisce una funzione anonima che incrementa e restituisce il valore di `i`, inizializzato a 0.  
La funzione restituita conserva accesso a quella specifica variabile `i`, quindi `nextInt() := intSeq()` non assegna un numero a `nextInt`, ma una funzione che incrementa e restituisce il valore di `i`. 

Con `newInts := intSeq()` si crea una nuova istanza, con un'altra variabile `i` --> la prima chiamata restituisce di nuovo 1.

Closure: es. 2 Fibonacci
```go
package main
import "fmt"

func fibonacci() func() int {
    x, y := 0, 1
    return func() int {
        x, y = y, x+y
        return x
    }
}

func main() {
    fib := fibonacci()
    for i := 0; i < 10; i++ {
        fmt.Println(fib())
    }
}
```
La closure aggiorna i valori utilizzando i valori vecchi delle variabili: soluzione elegante ed efficiente. Ad ogni chiamata si restituisce il numero successivo della sequenza di Fibonacci, per cui la closure mantiene lo stato tra le chiamate.

#### Metodi e interfacce
Go non ha classi, ma permette di associare **metodi** ai tipi.  
Un metodo è una funzione con parametro speciale, detto **receiver**, scritto tra `func` e il nome del metodo.

```go
type Vertex struct {
    X, Y float64
}

func (v Vertex) Abs() float64 {
    return math.Sqrt(v.X*v.X + v.Y*v.Y)
}
```
nella dichiarazione `func (v Vertex) Abs() float64` il receiver `v` è di tipo `Vertex`, `Abs` è il nome del metodo e restituisce un `float64`.

Il metodo si richiamerà come `v.Abs()` dove `v` è una variabile di tipo `Vertex`, e calcola $\sqrt{X^2 + Y^2}$ i.e. la distanza del punto dall'origine.  

Il receiver permette al metodo di accedere ai dati del valore sul quale viene chiamato.  
Un parallelismo con Java è che **la struct descrive i dati e il metodo definisce un comportamento associato a quel tipo**, la differenza è che in Go non c'è ereditarietà e tutto l'overhead della OOP classica.

Un'**interfaccia** descrive un insieme di **firme di metodi** richiesti.  
La firma specifica nome, parametri e ritorno di un metodo senza il corpo.

In Go un tipo soddisfa un'interfaccia se possiede tutti i metodi richiesti con le firme corrette, non serve quindi dichiarare `implements` come in Java. Il punto è infatti che in Go l'idea è costruire astrazioni basate sul comportamento, non sulla struttura dei dati.

Una funzione che richiede qualcosa capace di calcolare area e perimetro quindi può lavorare con qualsiasi tipo che implementi i metodi `Area()` e `Perimeter()`, senza preoccuparsi la rappresentazione interna dei dati, qui si vede il concetto di **polimorfismo**.

esempio:
```go
package main
import (
    "fmt"
    "math"
)

//define the interface for geometric space
type geometry interface {
    area() float64
    perim() float64
}

//implement the interface on rectangular and circle types
//(to implement an interface in Go, we must
//implement all the methods of the interface)
type rect struct {
    width, height float64
}
type circle struct {
    radius float64
}

//implement area and perim methods on rect
func (r rect) area() float64 {
    return r.width * r.height
}
func (r rect) perim() float64 {
    return 2*r.width + 2*r.height
}

//implement area and perim methods on circle
func (c circle) area() float64 {
    return math.Pi * c.radius * c.radius
}
func (c circle) perim() float64 {
    return 2 * math.Pi * c.radius
}
```
Abbiamo implementato l'interfaccia `geometry` su due tipi `rect` e `circle` --> entrambi soddisfano automaticamente l'interfaccia.

A questo punto possiamo scrivere una funzione che accetta un parametro di tipo `geometry` e quindi può lavorare con qualsiasi tipo che implementi quell'interfaccia:
```go
func measure(g geometry) {
    fmt.Println(g)
    fmt.Println(g.area())
    fmt.Println(g.perim())
}
func main() {
    r := rect{width: 3, height: 4}
    c := circle{radius: 5}

    measure(r)
    measure(c)
}
```
`measure(g geometry)` accetta un valore utilizzabile attraverso l'interfaccia `geometry`, quindi può essere chiamata con un `rect` o un `circle`. A quel punto si possono chiamare i metodi implementati da ciascun tipo, senza preoccuparsi della loro struttura interna.

## Concorrenza e goroutine
La concorrenza è una caratteristica centrale in Go. Le **goroutine** permettono di avviare attività concorrenti, mentre i **canali** e strumenti di sincronizzazione come `select` permettono di coordinare le attività e scambiare dati tra loro.

Concorrenza e parallelismo sono correlati ma diversi: la **concorrenza** permette di eseguire più attività alternandole, mentre il **parallelismo** permette di eseguire più attività contemporaneamente su più core.

Per avviare una goroutine basta anteporre la parola chiave `go` a una chiamata di funzione, es `go f(x, y, z)` avvia la funzione `f` e il chiamante prosegue senza aspettare che `f` termini.  
Le goroutine condividono lo spazio di indirizzamento del processo --> se accedono agli stessi dati e almeno una li modifica bisogna coordinarle correttamente.

Attenzione al fatto che **quando `main` termina, termina anche il programma anche se le altre goroutine non hanno ancora concluso --> avviare una goroutine non obbliga il `main` ad aspettarla**.

Le goroutine sono gestite dal runtime Go e vengono distribuite sui thread del SO. Il rapporto non è 1:1 --> **molte goroutine possono essere gestite attraverso un insieme più piccolo di thread**.  
Anche **l'ordine di esecuzione non è garantito**: avviare tre goroutine in ordine non significa che termineranno nello stesso.

### Canali
un **canale** consente alle goroutine di scambiarsi valori in modo coordinato. 

Il canale è tipizzato: `chan int` è un canale che trasporta valori di tipo `int`, `chan string` trasporta stringhe, ecc..  
Il canale viene creato con `make` (es. `ch := make(chan int)`). 

Si richiama la regola vista all'inizio sul verso della freccia: `ch <- v` invia il valore `v` al canale `ch`, mentre `v := <-ch` riceve un valore dal canale `ch` e lo assegna a `v`. Invece `<-ch` riceve un valore dal canale e lo scarta.

Un normale `chan T` consente sia invio che ricezione, e più goroutine possono usare lo stesso canale. Tuttavia una singola comunicazione consegna il valore **a un solo ricevente**, non è una trasmissione automatica a tutti.  
Il runtime gestisce la sincronizzazione interna sul canale --> le operazioni sul canale sono sicure, ma eventuali dati condivisi non sono necessariamente protetti.

Con `make(chan int)` si crea un canale **senza buffer** per cui l'invio blocca la goroutine fino a quando un'altra goroutine non riceve il valore, e la ricezione blocca la goroutine fino a quando un'altra goroutine non invia un valore.

Vediamo un esempio che divide la somma in due attività concorrenti:

<img src="imgs/go9.png" alt="go9" width="400"/>

`sum` riceve una slice e un canale, e calcola la somma locale per poi inviarla con `c <- sum` --> il risultato è comunicato tramite il canale, non tramite `return`.

Il `main` divide la slice in due: `s[:len(s)/2]` e `s[len(s)/2:]` (omettendo l'estremo iniziale intende 0, omettendo l'estremo finale intende la lunghezza della slice).  
Poi fa le due chiamate con `go`, che eseguono lo stesso codice su dati diversi, è esattamente il modello **SPMD (Single Program Multiple Data)**.  
Infine l'istruzione `sum1, sum2 := <-c, <-c` riceve i due risultati dal canale e poi si sommano in print.  
**Non è garantito quale somma arrivi prima**, e la sincronizzazione è ottenuta tramite ricezione senza lock esplicito.

Altro esempio è Fibonacci con una goroutine e un canale:

<img src="imgs/go10.png" alt="go10" width="400"/>

La funzione `fib` conserva `x` e `y` inizializzati a 1, ed esegue il ciclo infinito dove invia `x, aggiorna la coppia con `x, y = y, x+y` e ripete.  
Nel `main` si crea il canale, si avvia `fib` come goroutine e riceve 10 numeri, stampandoli.

Dal momento che il canale non ha buffer, il produttore e il consumatore si coordinano ad ogni scambio --> `fib` non può inviare indefinitamente senza che il `main` riceva.

Il programma si ferma dopo le 10 ricezioni, in quanto a quel punto il `main` termina --> la goroutine in questo caso non ha un proprio meccanismo di arresto, lo vediamo con `select`.


#### Canali buffered, chiusura dei canali, select
Sono da distinguere due tipi di canali:
- **unbuffered**: per crearlo `make(chan int)`, l'invio si blocca finché non c'è ricevente pronto e la ricezione si blocca finché non c'è un valore pronto da ricevere.
- **buffered**: per crearlo `make(chan int, 100)` con secondo argomento la capacità del buffer. L'invio si blocca solo se il buffer è pieno, la ricezione si blocca solo se il buffer è vuoto e il canale è ancora aperto.

Con buffer il produttore può quindi proseguire anche se il consumatore non riceve immediatamente, fintanto che c'è spazio disponibile rispetto ad altre goroutine che inviano sullo stesso canale.  
Ciò **disaccoppia parzialmente produttore e consumatore** dato che non devono sincronizzarsi ad ogni scambio.

Un canale senza buffer **ha capacità 0**, un canale con capacità 1 invece può conservare un solo elemento in attesa e così via.

Per chiudere un canale, segnalando quindi che non saranno inviati altri valori, si usa `close(ch)`. La chiusura non elimina gli elementi in buffer, i riceventi potranno ancora recuperarli.

Con `v, ok := <-ch` si riceve un valore dal canale e si verifica se il canale è chiuso. Se `ok` è `false`, allora il canale è chiuso e non ci sono più valori da ricevere, se `true` è stato ricevuto un valore valido.  
In particolare se il canale è chiuso e svuotato allora la ricezione restituisce lo zero value del tipo e `false`.

Inviare su un canale chiuso provoca un **panic**, per questo la chiusura deve essere fatta dal produttore (che sa quando gli invii sono terminati), non dal consumatore.  
Se ci sono più mittenti devono coordinarsi, non possono chiudere il canale indipendentemente.

Per ricevere e processare ripetutamente valori del canale si usa `range`:
```go
for i := range c {
    fmt.Println(i)
}
```
il ciclo termina quando il canale è **chiuso e svuotato**. Se è solo vuoto ma ancora aperto, resta in attesa --> **un canale vuoto non implica che la comunicazione sia conclusa**.

Per attendere tra più possibili invii o ricezioni, si usa `select`:
```go
select {
    case msg1 := <-ch1:
        //handle msg1
    case msg2 := <-ch2:
        //handle msg2
}
```
i due casi attendono un msg rispettivamente da `ch1` e `ch2`. Se nessuna ricezione può procedere --> la goroutine resta bloccata, mentre quando una può procedere viene eseguita con il relativo corpo.  
Se più casi sono pronti, **ne viene scelto uno (pseudo) u.a.r.**. 

Quindi la singola esecuzione di `select` sceglie un solo caso --> per continuare a gestire eventi infatti lo si inserisce normalmente in un ciclo infinito.  
`select` è utile per il **multiplexing** (i.e. gestire più comunicazioni nello stesso punto) e per attendere uan risposta insieme a un possibile timeout.

Es. Fibonacci con segnale di terminazione

```go
package main
import "fmt"

func fibonacci(c, quit chan int) {
    x, y := 1, 1
    for {
        select {
            case c <- x:    //send Fibonacci number
                x, y = y, x+y 
            case <-quit:    //receive termination signal
                fmt.Println("quit")
                return
        }
    }
}
```
In questa versione si aggiunge un secondo canale; mentre `ch` trasporta i numeri, `quit` serve a segnalare la terminazione.  
La funzione `fibonacci` esegue un ciclo infinito con due casi: se può inviare `x` sul canale dei dati --> lo invia e aggiorna la coppia, se riceve su `quit` allora stampa "quit" e termina con `return`.

```go 
func main() {
    ch := make(chan int) //unbuffered channel
    quit := make(chan int) 
    go func() {  //anonymous goroutine
        for i := 0; i < 10; i++ {
            fmt.Println(<-ch) //receive Fibonacci number
        }
        quit <- 0 //send termination signal
    }()
    fibonacci(ch, quit)
}
```
Il `main` crea i due canali e avvia una funct anonima come goroutine, questa riceve e stampa 10 numeri, e poi invia 0 su `quit`.  
La sintassi `go func() { ... }()` definisce una funzione anonima e ne avvia la chiamata in una nuova goroutine; le parentesi finali `()` sono necessarie per chiamare la funzione anonima.

Il `main` esegue poi `fibonacci(ch, quit)` senza il `go` in quanto il generatore gira sulla goroutine principale.   
Quello che succede praticamente è quanto segue: la goroutine principale produce numeri tramite `fibonacci`, che vengono ricevuti e stampati dalla goroutine anonima. Dopo 10 numeri, la goroutine anonima invia un segnale di terminazione su `quit`, che fa terminare la funzione `fibonacci` e quindi il programma.

Vediamo ora un esempio con `select` non bloccante tramite `default`:

```go
select {
    case msg1 := <-ch1:
        //handle msg1
    case msg2 := <-ch2:
        //handle msg2
    case ch3 <- msg3:
        //send msg3
    default:
        // ...
}
```
Qui se una comunicazione può procedere --> ne viene selezionata una. Se nessuna può procedere viene eseguito immediatamente `default`, che rende quindi **non bloccante la scelta**, senza di esso `select` dovrebbe aspettare.  

Inserire un `select` con un `default` vuoto in un ciclo rapido può far consumare CPU, dato che si controllano continuamente i canali.

Per aspettare si utilizzano i **timer channels**:

```go 
//to wait 2 seconds
timer := time.NewTimer(2 * time.Second)
    <- timer.C //wait for the timer to expire
```
Qui `time.NewTimer(2 * time.Second)` crea un timer che scade dopo 2 secondi.  
Quando si crea un timer, Go dà già il canale su cui ricevere la notifica di scadenza, accessibile con `timer.C`. L'istruzione `<- timer.C` blocca la goroutine fino a quando il timer non scade e invia un valore sul canale.

#### Mutex

Vediamo di realizzare didatticamente un **mutex** tramite un canale.  
Un mutex permette di proteggere una sezione critica, i.e. un blocco di codice che accede a dati condivisi e che deve essere eseguito da una sola goroutine alla volta.

```go
type Lock struct {
    ch chan bool
}
func NewLock() Lock {
    lock := Lock{ch: make(chan bool, 1)} //buffered channel with capacity 1
    lock.ch <- true //send
    return lock
}
func (l *Lock) Lock() {
    <-l.ch //receive
}
func (l *Lock) Unlock() {
    l.ch <- true //send
}
```
Lo struct `Lock` contiene un canale booleano con capacità 1.  
`NewLock` lo crea e vi inserisce `true`, questo valore funge da token che rappresenta la disponibilità del lock.

Il metodo `Lock()` riceve dal canale: se il token è presente, lo preleva e ottiene l'accesso. Se invece è assente, resta bloccato finché qualcuno non lo restituisce.  
Il metodo `Unlock()` invia nuovamente il gettone per rendere disponibile il lock.

Questo è solo un esempio didattico, nell'uso normale di Go si preferisce `sync.Mutex` che è più efficiente e sicuro.  
`sync.Mutex` offre `Lock()` e `Unlock()` per proteggere una sezione critica.

`sync.RWMutex` è una variante che permette di avere più lettori contemporaneamente, ma solo un singolo scrittore alla volta.  
Con `Lock() / Unlock()` si ottiene un lock esclusivo anche in termini di lettura per lo scrittore, mentre con `RLock() / RUnlock()` si ottiene un lock condiviso per la lettura; può essere utile quando le letture sono frequenti e le scritture rare.

`sync.WaitGroup` permette di attendere il completamento di un insieme di attività, questo risolve ad es. il problema del `main` che termina prima delle goroutine avviate.

Un esempio dell'utilizzo di `sync.WaitGroup` è per attività di tipo **BSP (Bulk Synchronous Parallel)**, dove attività indipendenti lavorano per una fase e poi si sincronizzano prima di passare alla fase successiva.  
Un'attività di questo tipo è la *discesa del gradiente*, per cui più lavoratori possono calcolare contributi su parti dei dati prima dell'aggiornamento globale che richiede di attendere tutti.

#### Priorità delle goroutine, moduli, dipendenze e ultime robe
Go non offre una normale API per assegnare priorità alle goroutine, scelta progettuale per dare semplicità allo scheduler.  

Se l'applicazione deve elaborare alcuni lavori prima di altri, bisogna organizzare la priorità nel codice, ad es. mediante 
- canali separati per diverse classi di priorità
- una coda di priorità, come un heap

Creare due canali ed inserirli in un `select` **non basta a privilegiare quello scritto per primo**, in quanto come visto se entrambi sono pronti viene scelto uno a caso.

Un **modulo** è una raccolta di package correlati, organizzati in una struttura di directory con un file `go.mod` alla radice.  
Si distinguono quindi:
- **package**, che usa codice utilizzabile tramite import
- **modulo**, che raggruppa package e ne descrive identità, versione Go, richiesta e dipendenze.

`$ go mod init <module-name>` crea un nuovo modulo con il nome specificato, e genera il file `go.mod`. Dentro `go.mod` si hanno:
- `module` che indica il percorso identificativo del modulo
- `go` che indica la versione minima di Go richiesta
- `require` che elenca dipendenze e versioni richieste

<img src="imgs/go11.png" alt="go11" width="300"/>

Con `$ go mod tidy` si aggiornano le dipendenze del modulo, rimuovendo quelle non più necessarie e aggiungendo quelle mancanti.

Una **funzione variadica** accetta un numero variabile di argomenti:
```go 
package main
import "fmt"

func sum(nums ...int) int { //nums is a slice of int
    fmt.Println(nums, " ")
    total := 0
    for _, num := range nums {
        total += num
    }
    fmt.Println(total)
}

func main() {
    sum(1, 2)
    sum(1, 2, 3)

    nums := []int{1, 2, 3, 4}
    sum(nums...) //unpack slice into variadic function
}
```
Il parametro `nums ...int` indica che la funzione accetta un numero variabile di argomenti di tipo `int`. Nella funzione, `nums` viene trattato come una slice di interi.

Il ciclo `range` attraversa i valori e ne calcola la somma, se gli argomenti si trovano già in una slice invece si usa `sum(nums...)` espandendo la slice nei singoli argomenti della chiamata.

**Gestione esplicita degli errori**: per convenzione in Go **l'errore è l'ultimo ritorno di una funzione**; il chiamante può controllare se è `nil` (nessun errore) o contiene un valore (errore).  
In questo modo la gestione degli errori è visibile nel normale flusso di codice, senza che sia spostata in meccanismi separati come la gestione delle eccezioni.

Il tipo built-in `error` è **un'interfaccia** che richiede il metodo `Error() string`. Una funzione che può generare un errore quindi restituisce un valore di tipo `error` come ultimo ritorno.

**Regole del codice e strumenti**: alcuni aspetti su cui Go è rigoroso:
- variabili locali dichiarate e non usate possono impedire la compilazione
- gli import inutilizzati generano errore
- alcune posizioni delle graffe come `} else {` sono richieste di sintassi

L'uso di `_` permette di scartare esplicitamente un risultato che non serve.

Alcuni strumenti utili:
- `gofmt` formatta il codice secondo le convenzioni di Go
- `goimports` gestisce gli import e formatta il codice
- `godoc` permette di consultare la documentazione

`gofmt -w yourcode.go` riscrive il file applicando la formattazione standard

Go offre il package `testing` e il comando `go test` per verificare il comportamento del codice. I file test terminano con `_test.go` e le funzioni test seguono la convenzione `func TestXxx(t *testing.T)` dove `Xxx` è un nome arbitrario che inizia con lettera maiuscola.

Un benchmark serve a misurare il costo di esecuzione di un'operazione, mentre il test ne verifica il comportamento.  
Anche i benchmark si trovano in file `_test.go` e seguono la convenzione `func BenchmarkXxx(b *testing.B)`.